In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import re
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

from common import (
    get_dataset_sort_key,
    INDEX_ORDER,
    index_colors,
    index_hatches,
    format_dataset_label,
)

# ---------------------------------------------------------------------------
# 1. Dataset properties and manually collected index size data (in bytes)
# ---------------------------------------------------------------------------
# Exact embedding counts and dimensions per dataset
DATASET_PROPS = {
    "Cohere (1M × 768":   {"n": 1_000_000, "d": 768},
    "OpenAI (500K × 1536": {"n": 500_000,   "d": 1536},
    "OpenAI (999K × 1536": {"n": 999_000,   "d": 1536},
    "Agnews (769K × 1024": {"n": 769_382,   "d": 1024},
}

# VSS (v1.4.4): CALL pragma_hnsw_index_info();
# pgvector (see docker-compose.yml): https://github.com/pgvector/pgvector/issues/276
# PDX (9df687b): CALL pdxearch_index_info();

# Structure: dataset_name → index_type → size in bytes
# Datasets match the case names used in index_creation.ipynb
data = {
    "Cohere (1M × 768": {
        "VSS HNSW":                          4558821312,
        "pgvector HNSW":                     None,
        "pgvector IVFFlat":                  None,
        "PDXearch SKM (IVF; Global; F32)":   3096384192,
        # "PDXearch SKM (IVF; Global; U8)":   792384208,
        "PDXearch SKM (IVF; Row Group; F32)": 3098223376,
        "PDXearch SKM (IVF; Row Group; U8)": 794223520,
    },
    "OpenAI (500K × 1536": {
        "VSS HNSW":                          4420603648,
        "pgvector HNSW":                     None,
        "pgvector IVFFlat":                  None,
        "PDXearch SKM (IVF; Global; F32)":   3095449464,
        # "PDXearch SKM (IVF; Global; U8)":   791449480,
        "PDXearch SKM (IVF; Row Group; F32)": 3093719152,
        "PDXearch SKM (IVF; Row Group; U8)": 789719232,
    },
    "OpenAI (999K × 1536": {
        "VSS HNSW":                          8853780608,
        "pgvector HNSW":                     None,
        "pgvector IVFFlat":                  None,
        "PDXearch SKM (IVF; Global; F32)":   6174503856,
        # "PDXearch SKM (IVF; Global; U8)":   1571111872,
        "PDXearch SKM (IVF; Row Group; F32)": 6177346416,
        "PDXearch SKM (IVF; Row Group; U8)": 1573954560,
    },
    "Agnews (769K × 1024": {
        "VSS HNSW":                          4556976368,
        "pgvector HNSW":                     None,
        "pgvector IVFFlat":                  None,
        "PDXearch SKM (IVF; Global; F32)":   3175078528,
        # "PDXearch SKM (IVF; Global; U8)":   811537040,
        "PDXearch SKM (IVF; Row Group; F32)": 3175191704,
        "PDXearch SKM (IVF; Row Group; U8)": 811650312,
    },
}

def format_bytes(n: float) -> str:
    """Format a byte count as a human-readable string."""
    if abs(n) < 1000:
        return f"{n:.0f} B"
    for unit in ('kB', 'MB', 'GB', 'TB'):
        n /= 1000
        if abs(n) < 1000:
            return f"{n:.2f} {unit}"
    return f"{n:.2f} PB"

In [ ]:
# ---------------------------------------------------------------------------
# 2. Dataset ordering
# ---------------------------------------------------------------------------

datasets = sorted(data.keys(), key=get_dataset_sort_key)

In [ ]:
# ---------------------------------------------------------------------------
# 3. Two bar plots: (1) absolute index size, (2) overhead vs. raw embeddings
# ---------------------------------------------------------------------------

# Styling (matching index_creation.ipynb)
mpl.rcParams['hatch.linewidth'] = 0.2
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']

label_fontsize = 14
tick_fontsize = 12
x_tick_fontsize = 12
bar_label_fontsize = 10
font_color = "#333333"
tick_fonts_color = '#585858'
bar_text_color = '#191919'

# Reference line styling
REF_F32_COLOR = '#333333'
REF_U8_COLOR  = '#333333'

# Which raw baseline to subtract for the overhead plot.
# F32 indexes are compared against raw F32, U8 indexes against raw U8.
INDEX_BASELINE = {
    "VSS HNSW":                          "f32",
    "pgvector HNSW":                     "f32",
    "pgvector IVFFlat":                  "f32",
    "PDXearch SKM (IVF; Global; F32)":   "f32",
    "PDXearch SKM (IVF; Row Group; F32)": "f32",
    "PDXearch SKM (IVF; Row Group; U8)": "u8",
}

# Determine which index types are actually present (non-None values)
all_index_types = []
for idx in INDEX_ORDER:
    for ds in datasets:
        if data[ds].get(idx) is not None:
            all_index_types.append(idx)
            break
n_index_types = len(all_index_types)

bar_width = 0.45
group_width = n_index_types * bar_width + 0.2
group_spacing = 0.4

from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

def bytes_formatter(val, pos):
    return format_bytes(val)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

def draw_bars(ax, value_fn, ylabel, draw_reflines=False):
    """Draw grouped bars on ax. value_fn(dataset, index_type, raw_f32, raw_u8) → float or None."""
    x_base = 0
    x_labels = []
    group_centers = []

    for dataset in datasets:
        group_start = x_base
        has_data = False

        props = DATASET_PROPS.get(dataset, {})
        n_emb = props.get("n", 0)
        dim   = props.get("d", 0)
        raw_f32 = n_emb * dim * 4
        raw_u8  = n_emb * dim * 1

        for idx_idx, index_type in enumerate(all_index_types):
            val = value_fn(dataset, index_type, raw_f32, raw_u8)
            if val is None:
                continue
            has_data = True
            x = group_start + idx_idx * bar_width
            color = index_colors.get(index_type, "#808080")
            hatch = index_hatches.get(index_type, "")
            ax.bar(x, val, bar_width, color=color, hatch=hatch,
                   label=index_type if dataset == datasets[0] else "")

            offset = max(1, abs(val) * 0.03)
            ax.text(x, val + offset, format_bytes(val),
                    ha="center", va="bottom", fontsize=bar_label_fontsize, color=bar_text_color)

        if has_data:
            center = group_start + (n_index_types - 1) * bar_width / 2
            group_centers.append(center)
            x_labels.append(format_dataset_label(dataset))

            if draw_reflines:
                x_left  = group_start - bar_width * 0.5
                x_right = group_start + (n_index_types - 1) * bar_width + bar_width * 0.5
                ax.hlines(raw_f32, x_left, x_right, colors=REF_F32_COLOR, linewidths=1.5,
                          linestyles='--',
                          label="Raw embeddings F32" if dataset == datasets[0] else "")
                ax.hlines(raw_u8, x_left, x_right, colors=REF_U8_COLOR, linewidths=1.5,
                          linestyles='--',
                          label="Raw embeddings U8" if dataset == datasets[0] else "")

            x_base += group_width + group_spacing

    ax.yaxis.set_major_formatter(FuncFormatter(bytes_formatter))
    ax.set_ylabel(ylabel, fontsize=label_fontsize, color=font_color)
    ax.set_xticks(group_centers)
    ax.set_xticklabels(x_labels, fontsize=x_tick_fontsize, color=tick_fonts_color)
    ax.yaxis.grid(True, linestyle='-', linewidth=0.6, color='gray', alpha=0.2)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left', 'bottom'):
        ax.spines[spine].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(axis='y', colors=tick_fonts_color)
    ax.tick_params(axis='both', length=0)
    ax.tick_params(axis='x', pad=5)

# Plot 1: absolute index size
def absolute_value(dataset, index_type, raw_f32, raw_u8):
    return data[dataset].get(index_type)

draw_bars(ax1, absolute_value, "In-Memory Index Size (Approximate Lower Bound)", draw_reflines=True)

# Plot 2: overhead = index size − raw embeddings size
def overhead_value(dataset, index_type, raw_f32, raw_u8):
    size = data[dataset].get(index_type)
    if size is None:
        return None
    baseline = raw_f32 if INDEX_BASELINE.get(index_type, "f32") == "f32" else raw_u8
    return size - baseline

draw_bars(ax2, overhead_value, "Index Overhead vs. Raw Embeddings", draw_reflines=False)

# Draw a zero-line on the overhead plot for reference
ax2.axhline(0, color='#333333', linewidth=0.8, linestyle='-', zorder=3)

# Shared legend
legend_handles = [
    Patch(facecolor=index_colors.get(idx, "#808080"),
          hatch=index_hatches.get(idx, ""),
          edgecolor='black', linewidth=0.5,
          label=idx)
    for idx in all_index_types
]
legend_handles += [
    Line2D([0], [0], color=REF_F32_COLOR, linewidth=1.5, linestyle='--', label="Raw embeddings F32"),
    Line2D([0], [0], color=REF_U8_COLOR,  linewidth=1.5, linestyle='--', label="Raw embeddings U8"),
]
fig.legend(handles=legend_handles, loc="upper center", ncols=4,
           frameon=False, fontsize=label_fontsize, bbox_to_anchor=(0.5, 1.06))

fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("index_size.pdf", dpi=300, bbox_inches='tight')